# 176. Second Highest Salary

[LeetCode problem](https://leetcode.com/problems/second-highest-salary/)


# 0. Problem

Return the second distinct highest salary. If no second distinct salary exists, return null.


# 1. Setup


In [ ]:
import pandas as pd
salary_rows=[(1,100),(2,200),(3,300),(4,300)]
salary_pd=pd.DataFrame(salary_rows,columns=['id','salary'])


In [ ]:
# In Colab if needed: !pip -q install pyspark
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window
spark=SparkSession.builder.getOrCreate()
schema=T.StructType([T.StructField('id',T.IntegerType(),False),T.StructField('salary',T.IntegerType(),False)])
salary_spark=spark.createDataFrame(salary_rows,schema)
salary_spark.createOrReplaceTempView('Employee')


# 2. SQL Solution


In [ ]:
sql_result=spark.sql("""SELECT MAX(salary) AS SecondHighestSalary FROM Employee WHERE salary < (SELECT MAX(salary) FROM Employee)""")
sql_result.show(truncate=False)


# 3. pandas Solution


In [ ]:
distinct_salaries=salary_pd['salary'].drop_duplicates().sort_values(ascending=False).tolist()
second_highest=distinct_salaries[1] if len(distinct_salaries)>=2 else None
pandas_result=pd.DataFrame({'SecondHighestSalary':[second_highest]})
pandas_result


# 4. PySpark Solution


In [ ]:
salary_window=Window.orderBy(F.desc('salary'))
ranked=salary_spark.select('salary').distinct().withColumn('salary_rank',F.dense_rank().over(salary_window))
spark_result=ranked.filter(F.col('salary_rank')==2).agg(F.max('salary').alias('SecondHighestSalary'))
spark_result.show(truncate=False)


# 5. Pattern Mapping

| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| distinct salary | max-below-max logic | `.drop_duplicates()` | `.distinct()` |
| second highest | outer `MAX` | sort and take second | `dense_rank()` rank 2 |
| missing second value | aggregate returns `NULL` | explicit `None` | aggregate on empty rank-2 set returns null |


# 6. Muscle-Memory Round


In [ ]:
# MUSCLE MEMORY — SQL
# Use temp view: Employee


In [ ]:
# MUSCLE MEMORY — PANDAS
# Use DataFrame: salary_pd


In [ ]:
# MUSCLE MEMORY — PYSPARK
# Use DataFrame: salary_spark
